两个基本概念，自由索引/自由标（Free indices）和求和索引/哑标（Summation indices）：

自由索引，出现在箭头右边的索引
求和索引，只出现在箭头左边的索引，表示中间计算结果需要这个维度上求和之后才能得到输出，

接着是介绍三条基本规则：

规则一，equation 箭头左边，在不同输入之间重复出现的索引表示，把输入张量沿着该维度做乘法操作，比如还是以上面矩阵乘法为例， "ik,kj->ij"，k 在输入中重复出现，所以就是把 a 和 b 沿着 k 这个维度作相乘操作；

规则二，只出现在 equation 箭头左边的索引，表示中间计算结果需要在这个维度上求和，也就是上面提到的求和索引；

规则三，equation 箭头右边的索引顺序可以是任意的，比如上面的 "ik,kj->ij" 如果写成 "ik,kj->ji"，那么就是返回输出结果的转置，用户只需要定义好索引的顺序，转置操作会在 einsum 内部完成。

两条特殊规则：

equation 可以不写包括箭头在内的右边部分，那么在这种情况下，输出张量的维度会根据默认规则推导。就是把输入中只出现一次的索引取出来，然后按字母表顺序排列，比如上面的矩阵乘法 "ik,kj->ij" 也可以简化为 "ik,kj"，根据默认规则，输出就是 "ij" 与原来一样；

equation 中支持 "..." 省略号，用于表示用户并不关心的索引，详见下方转置例子

获取对角线元素diagonal

In [2]:
import torch

a = torch.arange(16).reshape(4,4)
torch.einsum('ii->i', a)

tensor([ 0,  5, 10, 15])

迹trace  省略 ΣΣ，左右两边对调，省去矩阵和 t，剩下的就是ii->

In [3]:
torch.einsum('ii',a)

tensor(30)

In [4]:
b = torch.randn(3,4,5,7)
torch.einsum('...ij->...ji',b)

tensor([[[[-1.0272,  0.3850, -1.6655,  0.4496, -1.6889],
          [-1.0824, -0.0683, -0.8751, -0.0227,  0.1668],
          [-2.0709,  0.9927, -0.5357, -0.1106, -1.0717],
          [ 0.5067, -0.8336, -0.1655, -0.1373,  2.1374],
          [ 1.4640,  0.3937,  0.4403, -0.3600, -1.5467],
          [-0.2566, -0.3661,  2.6702,  0.1078, -0.6929],
          [ 0.5471, -1.4811, -1.1256,  0.0579,  0.8973]],

         [[-0.9233,  1.1517, -1.7679,  1.4183,  0.4035],
          [ 0.3638,  1.6596,  0.4613, -0.8058,  1.2381],
          [-0.7328,  0.5133,  0.2209,  0.1193, -0.2960],
          [-0.5191, -1.7846,  0.7375, -1.5188,  0.7275],
          [ 1.6060,  0.1905,  0.9986, -1.1387,  0.7206],
          [ 1.1881,  0.0205,  0.5072, -1.1986, -1.7826],
          [ 0.2175,  0.0234, -0.7881, -0.1809, -1.1422]],

         [[ 0.0261,  0.2313, -0.9384, -0.5916, -0.9970],
          [-0.0336, -0.7417, -0.5377,  0.0092,  0.6342],
          [ 0.1734, -0.6962,  1.1004, -2.1342,  0.0505],
          [-1.8552,  0.0790

求和

In [6]:
a = torch.arange(6).reshape(2, 3)
torch.einsum('ij->', [a])


tensor(15)

矩阵乘法系列

In [7]:
a = torch.randn(2,3)
b = torch.randn(3,4)
c = torch.matmul(a,b)
c = torch.einsum('ik,kj->ij', a, b)
c

tensor([[-0.4994, -1.4934,  2.3036,  1.7716],
        [ 0.0858, -0.1443, -0.4328, -0.1177]])

In [17]:
a = torch.randn(2,3)
b = torch.arange(3).to(a.dtype)

torch.einsum('ik,k->i', a, b)

tensor([-0.5909,  0.7689])

多头注意力机制

In [20]:
# q k v均为 (b,nums_head, s, h)
q = torch.randn(2, 3, 4, 5)
k = torch.randn(2, 3, 4, 5)

c = torch.einsum('bnqh,bnkh->bnqk', q, k)
print(c.shape)

torch.Size([2, 3, 4, 4])


广播机制规则

2.1 如果遵守以下规则，则两个tensor是“可广播的”：

每个tensor至少有一个维度；

遍历tensor所有维度时，从末尾开始遍历（从右往左开始遍历）（从后往前开始遍历），两个tensor存在下列情况：

tensor维度相等。

tensor维度不等且其中一个维度为1。

tensor维度不等且其中一个维度不存在。

2.2 如果两个tensor是“可广播的”，则计算过程遵循下列规则：

如果两个tensor的维度不同，则在维度较小的tensor的前面增加维度，使它们维度相等。

对于每个维度，计算结果的维度值取两个tensor中较大的那个值。

两个tensor扩展维度的过程是将数值进行复制。

In [21]:
# x 和 y 可以广播
x=torch.ones(5,3,4,1)
y=torch.ones(  3,1,1)
z = x+y
x.shape,y.shape,z.shape
# 从尾部维度开始遍历
# 1st尾部维度: x和y相同，都为1。
# 2nd尾部维度: y为1，x为4,符合维度不等且其中一个维度为1，则广播为4。
# 3rd尾部维度: x和y相同，都为3。
# 4th尾部维度: y维度不存在，x为5,符合维度不等且其中一个维度不存在，则广播为5。


(torch.Size([5, 3, 4, 1]), torch.Size([3, 1, 1]), torch.Size([5, 3, 4, 1]))

矩阵吸收 torch.einsum 是支持广播的

先把 query 的最后一维 d（大维度，比如 256）投影到一个低维空间 c（比如 64）。

每个 head 有自己独立的投影矩阵（[h, d, c]），不会相互干扰。

这一步就是 矩阵吸收：把“大矩阵”里的部分权重“吸收”到每个 head 的小投影里，降低计算开销和参数规模。

In [ ]:
q_nope = torch.randn(2, 3, 4, 100)
wkv_b = torch.randn(4, 5, 10)
q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b)
q_nope.shape

torch.Size([2, 3, 4, 10])

MLA中的注意力机制

In [ ]:
import torch

q_nope = torch.tensor([[
    [[1.0, 2.0, 3.0]],   # 第一个 query (s=0)
    [[4.0, 5.0, 6.0]]    # 第二个 query (s=1)
]])  # shape [1,2,1,3]

kv_cache = torch.tensor([[
    [1.0, 0.0, 1.0],     # 第一个 key
    [0.0, 1.0, 1.0]      # 第二个 key
]])  # shape [1,2,3]

scores_nope = torch.einsum("bshc,btc->bsht", q_nope, kv_cache)  # bshc btc -> bshc bct -> bsht
print(scores_nope)
print(scores_nope.shape)


tensor([[[[ 4.,  5.]],

         [[10., 11.]]]])
torch.Size([1, 2, 1, 2])
